# Gradient Boosting — Water Quality Targets

Trains a **histogram-based gradient boosting regression** model
(`HistGradientBoostingRegressor`) for each of twelve water-quality target
variables using the terminal modeling table
`data/03c_merge_tertiary/epa-full.csv`, then evaluates every model on a held-out
test split and reports **R²**, **RMSE**, and **Error Rate** (symmetric MAPE, %).

This is the boosted-tree counterpart to `multiple_linear_regression.ipynb` and
`random_forest.ipynb` — the targets, features, split, and metrics are identical,
so all three notebooks are directly comparable.

**Targets modeled:** water temperature, dissolved oxygen, pH, nitrate, nitrite,
nitrate + nitrite, total phosphorus, specific conductance, total dissolved
solids, total suspended solids, turbidity, and *E. coli*.

Each model is a scikit-learn `Pipeline`:

1. `SimpleImputer(strategy="median")` — fill missing predictor values
2. `HistGradientBoostingRegressor(...)` — the boosted-tree estimator

No feature scaling is needed (the model is invariant to monotone rescaling).
`HistGradientBoostingRegressor` can in fact consume `NaN` natively, but we keep
the median imputer so preprocessing matches the other two notebooks exactly and
the model comparison stays apples-to-apples.

The notebook is split into parts:

* **Part 1 — Train** every model and keep its held-out test split.
* **Part 2 — Test** every trained model and summarize the three metrics.
* **Part 3 — Feature importances** (permutation-based) for interpretability.

---

**How the data is split.** `GroupShuffleSplit(test_size=0.2, random_state=42)`
grouped on `MonitoringLocationIdentifier`: 20% of the monitoring stations are
held out **whole**, so no station appears on both sides of the split and every
score below answers *"how well does this predict at a station the model has
never seen?"*.

This replaces the earlier random row split, which left 99.1–99.7% of test rows
at a station that was also in the training set. With latitude and longitude as
predictors — and roughly one distinct coordinate pair per station — a tree could
identify the station and recall its typical level, so the old scores measured
recall as much as prediction (`src/04_eda/eda-summary.md`, red flags 1–2).

**Every R² is reported beside a persistence baseline** — "repeat this station's
previous value", a model with no features at all — measured on the same held-out
rows. The margin between them is how much the 30 predictors actually buy.

In [1]:
# --- Imports ---
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
RANDOM_STATE = 42
TEST_SIZE = 0.2               # share of *stations* held out — not share of rows
MIN_SAMPLES = 100             # skip a target with fewer usable rows than this
MIN_PERSISTENCE_PAIRS = 30    # below this the persistence baseline is not reported

# The train/test split is grouped on the station id: no monitoring location may
# appear on both sides. A plain random row split put 99%+ of test rows at a
# station that was also in training, and latitude/longitude are near-unique per
# station (~1 distinct value each per station) — so a tree can recover the
# station from its coordinates and recall its typical level. That inflates every
# score. See src/04_eda/eda-summary.md, red flags 1-2.
GROUP_COL = "MonitoringLocationIdentifier"
DATE_COL = "ActivityStartDateTime"

# HistGradientBoosting hyperparameters (shared across all targets).
GB_PARAMS = dict(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=500,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=0.1,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
)

## Configuration

Locate the dataset, and declare the targets and predictor features.

In [2]:
# --- Locate the repo root and the terminal modeling table ---
# The notebook may be launched from anywhere; walk upward until we find the CSV.
def find_data_path() -> Path:
    rel = Path("data/03c_merge_tertiary/epa-full.csv")
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / rel
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {rel} by walking up from {here}. "
        "Run the notebook from within the repository."
    )

DATA_PATH = find_data_path()
print("Using dataset:", DATA_PATH)

# Repo root — where the .pkl files and model_metrics.csv are written.
REPO_ROOT = DATA_PATH.parents[2]
print("Repo root:", REPO_ROOT)

Using dataset: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/data/03c_merge_tertiary/epa-full.csv
Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction


In [3]:
# --- Target variables: label -> (CSV column, plausible valid range) ---
# The valid range drops physically impossible readings and data-entry errors
# before fitting (e.g. a pH of 999 or a negative concentration).
TARGETS = {
    "Water Temperature":      {"column": "Temperature, water_value",            "valid_range": (-5.0, 45.0)},
    "Dissolved Oxygen":       {"column": "Dissolved oxygen (DO)_value",         "valid_range": (0.0, 30.0)},
    "pH":                     {"column": "pH_value",                            "valid_range": (0.0, 14.0)},
    "Nitrate":                {"column": "Nitrate_value",                       "valid_range": (0.0, 100.0)},
    "Nitrite":                {"column": "Nitrite_value",                       "valid_range": (0.0, 20.0)},
    "Nitrate + Nitrite":      {"column": "Nitrate + Nitrite_value",             "valid_range": (0.0, 100.0)},
    "Total Phosphorus":       {"column": "Total Phosphorus, mixed forms_value", "valid_range": (0.0, 25.0)},
    "Specific Conductance":   {"column": "Specific conductance_value",          "valid_range": (0.0, 10000.0)},
    "Total Dissolved Solids": {"column": "Total dissolved solids_value",        "valid_range": (0.0, 10000.0)},
    "Total Suspended Solids": {"column": "Total suspended solids_value",        "valid_range": (0.0, 10000.0)},
    "Turbidity":              {"column": "Turbidity_value",                     "valid_range": (0.0, 5000.0)},
    "E. coli":                {"column": "Escherichia coli_value",              "valid_range": (0.0, 1_000_000.0)},
}

# --- Predictor features ---
# Environmental / spatial / temporal drivers only. We deliberately exclude the
# other water-quality "_value" columns so a model never predicts one target
# from another measured target.
BASE_FEATURE_COLS = [
    # Location
    "LatitudeMeasure", "LongitudeMeasure",
    "distance_to_climate_station_km", "distance_to_streamflow_gauge_km",
    # PRISM climate normals at the observation
    "prism_tmax_c", "prism_tmin_c", "prism_ppt_mm", "prism_tdmean_c",
    # ISU station weather
    "isu_avg_wind_speed_kts", "isu_avg_rh", "isu_snow_in", "isu_snowd_in",
    "isu_max_feel_c", "isu_min_feel_c",
    # Hydrology
    "streamflow_discharge_cfs",
    # Soil
    "ksat_mean", "awc_mean",
    # Land cover
    "pct_corn", "pct_soybean", "pct_developed", "pct_forest", "pct_row_crops",
    # Nutrient loading context
    "npfert__n__total_kg", "npfert__p__total_kg",
    "npmanure__total__n_kg", "npmanure__total__p_kg",
]

# Temporal features engineered from the timestamp (added in the next cell).
TEMPORAL_FEATURE_COLS = ["doy", "doy_sin", "doy_cos", "obs_year"]

FEATURE_COLS = BASE_FEATURE_COLS + TEMPORAL_FEATURE_COLS
print(f"{len(FEATURE_COLS)} predictor features")

30 predictor features


## Load and prepare the data

Parse the timestamp and derive seasonal (day-of-year) features.

In [4]:
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)

    # Derive temporal predictors from the activity timestamp. Build them as one
    # block and concat once, so we don't fragment the already-wide frame.
    ts = pd.to_datetime(df["ActivityStartDateTime"], errors="coerce")
    doy = ts.dt.dayofyear
    radians = 2.0 * np.pi * doy / 365.25  # cyclical: day 365 sits next to day 1
    temporal = pd.DataFrame({
        "doy": doy,
        "obs_year": ts.dt.year,
        "doy_sin": np.sin(radians),
        "doy_cos": np.cos(radians),
        # Not a predictor — the ordering key for the persistence baseline.
        "_obs_ts": ts,
    }, index=df.index)
    return pd.concat([df, temporal], axis=1)


data = load_dataset(DATA_PATH)
print("Rows:", len(data), "| Columns:", data.shape[1])

# Sanity-check every declared predictor actually exists.
missing = [c for c in FEATURE_COLS if c not in data.columns]
assert not missing, f"Missing predictor columns: {missing}"
data[FEATURE_COLS].describe().T[["count", "mean", "std", "min", "max"]]

Rows: 48251 | Columns: 323


,count,mean,std,min,max
LatitudeMeasure,"48,251.0000",41.8428,0.7139,40.3871,43.5002
LongitudeMeasure,"48,251.0000",-93.0813,1.4052,-96.6325,-90.2010
distance_to_climate_station_km,"48,251.0000",21.5023,12.0707,0.2631,74.5673
distance_to_streamflow_gauge_km,"48,251.0000",5.0970,4.8408,0.0000,25.8260
prism_tmax_c,"46,602.0000",21.8390,9.8877,-20.4180,38.9453
prism_tmin_c,"46,632.0000",10.1485,9.2721,-29.3780,27.0850
prism_ppt_mm,"46,629.0000",3.3790,9.4365,0.0000,131.6350
prism_tdmean_c,"46,625.0000",10.8877,9.3835,-29.1303,27.2638
isu_avg_wind_speed_kts,"45,079.0000",7.0451,3.5216,0.0000,25.5105
isu_avg_rh,"44,806.0000",72.6710,12.8659,1.0227,100.0000


## Metrics

* **R²** — coefficient of determination on the held-out test set.
* **RMSE** — root mean squared error, in the target's own units.
* **Error Rate** — symmetric mean absolute percentage error (sMAPE), reported as
  a percentage. sMAPE is bounded and stays well-behaved when the true value is
  near zero, which matters for skewed concentration targets. This matches the
  `error_rate_pct` convention used elsewhere in the repo.


In [5]:
def symmetric_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric MAPE as a percentage (0 = perfect). Robust to y_true near 0."""
    denom = np.abs(y_true) + np.abs(y_pred)
    numer = 2.0 * np.abs(y_true - y_pred)
    safe = np.divide(numer, denom, out=np.zeros_like(denom, dtype=float), where=denom != 0)
    return float(np.mean(safe) * 100.0)


def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "Error Rate (%)": symmetric_mape(y_true, y_pred),
    }

## Part 1 — Train every model

For each target we drop rows with no measurement, clip to the valid range, then
split **by station** with `GroupShuffleSplit(test_size=0.2)` grouped on
`MonitoringLocationIdentifier`: 20% of the *stations* that measured this target
are held out whole, and none of their rows are ever seen during training. The
row share of the test set therefore drifts away from 20% — station volume is
heavy-tailed (the median station has 7 observations, the busiest 2,878) — so the
actual row and station counts are printed per target.

This answers "can the model predict at a station it has never seen?". It is a
harder and more honest question than the random row split this notebook used
previously, which held out rows from stations it had already memorised. Expect
the R² of the station-dominated targets (Specific Conductance, Total Dissolved
Solids) to fall the furthest.

Trained models and their splits are cached in `TRAINED` for the testing section
below.

In [6]:
def make_pipeline() -> Pipeline:
    """Impute -> histogram gradient boosting."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(**GB_PARAMS)),
    ])


def prepare_frame(df: pd.DataFrame, column: str, valid_range: tuple) -> pd.DataFrame:
    """Rows usable for one target: predictors + target + station id + timestamp.

    Drops rows with no measurement and clips to the plausible range. The station
    id and timestamp ride along because the split is grouped by station and the
    persistence baseline needs each station's observations in time order.
    """
    frame = (
        df[FEATURE_COLS + [column, GROUP_COL, "_obs_ts"]]
        .dropna(subset=[column, GROUP_COL])
        .copy()
    )
    lo, hi = valid_range
    return frame[frame[column].between(lo, hi)]


def grouped_split(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Hold out whole stations — no station may straddle the train/test boundary."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(frame, groups=frame[GROUP_COL]))
    return frame.iloc[train_idx], frame.iloc[test_idx]


TRAINED: dict[str, dict] = {}

for label, cfg in TARGETS.items():
    column = cfg["column"]
    frame = prepare_frame(data, column, cfg["valid_range"])
    if len(frame) < MIN_SAMPLES:
        print(f"[SKIP] {label}: only {len(frame)} usable rows (< {MIN_SAMPLES}).")
        continue

    train, test = grouped_split(frame)
    train_stations = set(train[GROUP_COL])
    test_stations = set(test[GROUP_COL])
    assert not (train_stations & test_stations), f"{label}: station leaked across the split"

    X_train = train[FEATURE_COLS].to_numpy(dtype=float)
    y_train = train[column].to_numpy(dtype=float)
    X_test = test[FEATURE_COLS].to_numpy(dtype=float)
    y_test = test[column].to_numpy(dtype=float)

    pipeline = make_pipeline()
    pipeline.fit(X_train, y_train)

    # HistGradientBoosting stops early; the other estimators have no round count.
    rounds = getattr(pipeline.named_steps["model"], "n_iter_", None)

    TRAINED[label] = {
        "column": column,
        "model": pipeline,
        "test": test,               # full frame — the persistence baseline needs it
        "X_test": X_test,
        "y_test": y_test,
        "n_total": len(frame),
        "n_train": len(train),
        "n_test": len(test),
        "n_train_stations": len(train_stations),
        "n_test_stations": len(test_stations),
    }
    print(f"[OK]   {label:24s} train {len(train):>6,} rows / {len(train_stations):>4,} stations"
          f"  |  test {len(test):>5,} rows / {len(test_stations):>4,} stations"
          f"  ({len(test) / len(frame):>4.0%} of rows)"
          + (f"  | rounds {rounds}" if rounds else ""))

print(f"\nTrained {len(TRAINED)} of {len(TARGETS)} target models. "
      "No station appears in both the training and the test set.")

[OK]   Water Temperature        train 28,613 rows /  799 stations  |  test 6,092 rows /  200 stations  ( 18% of rows)  | rounds 500


[OK]   Dissolved Oxygen         train 25,185 rows /  727 stations  |  test 6,640 rows /  182 stations  ( 21% of rows)  | rounds 500


[OK]   pH                       train 26,680 rows /  885 stations  |  test 5,673 rows /  222 stations  ( 18% of rows)  | rounds 500


[OK]   Nitrate                  train  9,628 rows /  222 stations  |  test 2,719 rows /   56 stations  ( 22% of rows)  | rounds 170
[OK]   Nitrite                  train  9,340 rows /  183 stations  |  test 2,271 rows /   46 stations  ( 20% of rows)  | rounds 28


[OK]   Nitrate + Nitrite        train  3,614 rows /  293 stations  |  test 1,045 rows /   74 stations  ( 22% of rows)  | rounds 173


[OK]   Total Phosphorus         train  4,592 rows /  362 stations  |  test 1,294 rows /   91 stations  ( 22% of rows)  | rounds 295


[OK]   Specific Conductance     train 11,450 rows /  386 stations  |  test 4,617 rows /   97 stations  ( 29% of rows)  | rounds 500


[OK]   Total Dissolved Solids   train 14,109 rows /  465 stations  |  test 3,902 rows /  117 stations  ( 22% of rows)  | rounds 500


[OK]   Total Suspended Solids   train 12,088 rows /  442 stations  |  test 2,435 rows /  111 stations  ( 17% of rows)  | rounds 129


[OK]   Turbidity                train 17,575 rows /  675 stations  |  test 3,581 rows /  169 stations  ( 17% of rows)  | rounds 145
[OK]   E. coli                  train 12,066 rows /  347 stations  |  test 3,831 rows /   87 stations  ( 24% of rows)  | rounds 26

Trained 12 of 12 target models. No station appears in both the training and the test set.


## Part 2 — Test every model

`test_model()` scores one cached model on its held-out test set and returns the
metrics. `persistence_baseline()` scores the zero-feature alternative — "repeat
this station's previous value" — on the same held-out rows, so every R² can be
read against the memorisation bar it has to clear. `test_all_models()` runs both
across every trained target and assembles a summary table.

In [7]:
def test_model(label: str, verbose: bool = True) -> dict:
    """Evaluate one trained model on its held-out test set."""
    if label not in TRAINED:
        raise KeyError(f"No trained model for {label!r}. Run Part 1 first.")
    entry = TRAINED[label]
    y_pred = entry["model"].predict(entry["X_test"])
    metrics = evaluate(entry["y_test"], y_pred)
    if verbose:
        print(f"{label}  (n_test={entry['n_test']:,} rows "
              f"from {entry['n_test_stations']:,} unseen stations)")
        print(f"    R2         = {metrics['R2']:.4f}")
        print(f"    RMSE       = {metrics['RMSE']:.4f}")
        print(f"    Error Rate = {metrics['Error Rate (%)']:.2f}%")
    return metrics


def persistence_baseline(label: str) -> dict:
    """Score "repeat this station's previous value" on the same held-out rows.

    Because the split is grouped by station, every observation a test station
    ever made sits in the test set — so this baseline is free to use the site's
    own history, which the model never saw. That makes it the memorisation bar:
    a model that cannot beat it is recalling the site rather than predicting the
    water. (Read the margin with the revisit gap in mind — a target resampled
    the next day is far easier to persist than one resampled a month later.)

    Both scores are computed on the *same* rows — those that have a previous
    observation at the same station — so the margin is like-for-like.
    """
    entry = TRAINED[label]
    column = entry["column"]
    ordered = entry["test"].sort_values([GROUP_COL, "_obs_ts"])
    by_station = ordered.groupby(GROUP_COL, observed=True)
    prev = by_station[column].shift(1).to_numpy(dtype=float)
    gap_days = by_station["_obs_ts"].diff().dt.days.to_numpy(dtype=float)
    paired = ~np.isnan(prev)

    out = {
        "n_pairs": int(paired.sum()),
        "median_gap_days": np.nan,
        "Persistence R2": np.nan,
        "Persistence RMSE": np.nan,
        "Model R2 (paired)": np.nan,
        "Model RMSE (paired)": np.nan,
        "Margin": np.nan,
    }
    if out["n_pairs"] < MIN_PERSISTENCE_PAIRS:
        return out

    rows = ordered[paired]
    y_true = rows[column].to_numpy(dtype=float)
    y_prev = prev[paired]
    y_pred = entry["model"].predict(rows[FEATURE_COLS].to_numpy(dtype=float))

    out["median_gap_days"] = float(np.median(gap_days[paired]))
    out["Persistence R2"] = float(r2_score(y_true, y_prev))
    out["Persistence RMSE"] = float(mean_squared_error(y_true, y_prev) ** 0.5)
    out["Model R2 (paired)"] = float(r2_score(y_true, y_pred))
    out["Model RMSE (paired)"] = float(mean_squared_error(y_true, y_pred) ** 0.5)
    out["Margin"] = out["Model R2 (paired)"] - out["Persistence R2"]
    return out


def test_all_models() -> pd.DataFrame:
    rows = []
    for label, entry in TRAINED.items():
        m = test_model(label, verbose=False)
        p = persistence_baseline(label)
        rows.append({
            "Target": label,
            "Column": entry["column"],
            "N test": entry["n_test"],
            "N test stations": entry["n_test_stations"],
            "R2": m["R2"],
            "RMSE": m["RMSE"],
            "MAE": m["MAE"],
            "Error Rate (%)": m["Error Rate (%)"],
            "Persistence R2": p["Persistence R2"],
            "Model R2 (paired)": p["Model R2 (paired)"],
            "Margin": p["Margin"],
        })
    summary = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
    return summary

### Per-model report

In [8]:
for label in TRAINED:
    test_model(label)
    print()

Water Temperature  (n_test=6,092 rows from 200 unseen stations)
    R2         = 0.9395
    RMSE       = 2.1685
    Error Rate = 23.05%

Dissolved Oxygen  (n_test=6,640 rows from 182 unseen stations)
    R2         = 0.4668
    RMSE       = 1.9616
    Error Rate = 16.67%

pH  (n_test=5,673 rows from 222 unseen stations)
    R2         = 0.3517
    RMSE       = 0.5101
    Error Rate = 4.69%

Nitrate  (n_test=2,719 rows from 56 unseen stations)
    R2         = 0.4155
    RMSE       = 4.3784
    Error Rate = 105.29%

Nitrite  (n_test=2,271 rows from 46 unseen stations)
    R2         = -0.1216
    RMSE       = 0.1297
    Error Rate = 180.37%

Nitrate + Nitrite  (n_test=1,045 rows from 74 unseen stations)
    R2         = 0.4998
    RMSE       = 3.1969
    Error Rate = 66.76%



Total Phosphorus  (n_test=1,294 rows from 91 unseen stations)
    R2         = 0.0266
    RMSE       = 0.4705
    Error Rate = 66.99%

Specific Conductance  (n_test=4,617 rows from 97 unseen stations)
    R2         = 0.1846
    RMSE       = 183.3914
    Error Rate = 28.19%

Total Dissolved Solids  (n_test=3,902 rows from 117 unseen stations)
    R2         = 0.4382
    RMSE       = 94.2563
    Error Rate = 22.53%

Total Suspended Solids  (n_test=2,435 rows from 111 unseen stations)
    R2         = -0.0357
    RMSE       = 212.2647
    Error Rate = 101.61%

Turbidity  (n_test=3,581 rows from 169 unseen stations)
    R2         = -0.0986
    RMSE       = 91.7846
    Error Rate = 94.59%

E. coli  (n_test=3,831 rows from 87 unseen stations)
    R2         = 0.1157
    RMSE       = 9483.7225
    Error Rate = 127.24%



### Summary table

All twelve models side by side, sorted by test R².

In [9]:
summary = test_all_models()
summary

,Target,Column,N test,N test stations,R2,RMSE,MAE,Error Rate (%),Persistence R2,Model R2 (paired),Margin
0,Water Temperature,"Temperature, water_value",6092,200,0.9395,2.1685,1.6219,23.0520,0.6403,0.9417,0.3014
1,Nitrate + Nitrite,Nitrate + Nitrite_value,1045,74,0.4998,3.1969,2.1952,66.7612,0.1896,0.4579,0.2683
2,Dissolved Oxygen,Dissolved oxygen (DO)_value,6640,182,0.4668,1.9616,1.4071,16.6675,0.3249,0.4744,0.1495
3,Total Dissolved Solids,Total dissolved solids_value,3902,117,0.4382,94.2563,69.2902,22.5271,0.8107,0.4345,-0.3762
4,Nitrate,Nitrate_value,2719,56,0.4155,4.3784,2.6519,105.2872,0.3984,0.4119,0.0136
5,pH,pH_value,5673,222,0.3517,0.5101,0.3683,4.6858,-0.0605,0.3536,0.4141
6,Specific Conductance,Specific conductance_value,4617,97,0.1846,183.3914,138.1098,28.1902,0.8559,0.1581,-0.6978
7,E. coli,Escherichia coli_value,3831,87,0.1157,"9,483.7225","2,058.3660",127.2434,-0.7406,0.1162,0.8568
8,Total Phosphorus,"Total Phosphorus, mixed forms_value",1294,91,0.0266,0.4705,0.2024,66.9868,0.3191,0.1029,-0.2162
9,Total Suspended Solids,Total suspended solids_value,2435,111,-0.0357,212.2647,72.7747,101.6110,-0.9931,-0.0401,0.9530


### Persistence baseline — how much of the score is memorisation?

A model with **no features at all** — "this station's next value equals its
previous value" — is the bar any of these models has to clear before the word
*prediction* applies. `Margin` is the model's R² minus the persistence R² on the
identical set of rows; a negative margin means 30 environmental predictors buy
less than repeating the last reading.

In [10]:
persistence = (
    pd.DataFrame([{"Target": label, **persistence_baseline(label)} for label in TRAINED])
    .set_index("Target")
    .sort_values("Margin")
)

view = persistence[["n_pairs", "median_gap_days", "Persistence R2",
                    "Persistence RMSE", "Model R2 (paired)", "Model RMSE (paired)",
                    "Margin"]]
print(view.round(3).to_string())

beaten = persistence["Margin"] > 0
print(f"\nBeats persistence on {int(beaten.sum())} of {int(beaten.notna().sum())} "
      "targets with a reportable baseline.")
view

                        n_pairs  median_gap_days  Persistence R2  Persistence RMSE  Model R2 (paired)  Model RMSE (paired)  Margin
Target                                                                                                                            
Specific Conductance       4520           1.0000          0.8560           75.4690             0.1580             182.4120 -0.6980
Total Dissolved Solids     3785          28.0000          0.8110           54.5530             0.4340              94.2790 -0.3760
Total Phosphorus           1203          33.0000          0.3190            0.3330             0.1030               0.3820 -0.2160
Nitrate                    2663          15.0000          0.3980            4.4060             0.4120               4.3560  0.0140
Dissolved Oxygen           6458          20.0000          0.3250            2.1920             0.4740               1.9340  0.1490
Nitrate + Nitrite           971          33.0000          0.1900            3.9840 

,n_pairs,median_gap_days,Persistence R2,Persistence RMSE,Model R2 (paired),Model RMSE (paired),Margin
Target,,,,,,,
Specific Conductance,4520,1.0000,0.8559,75.4694,0.1581,182.4117,-0.6978
Total Dissolved Solids,3785,28.0000,0.8107,54.5535,0.4345,94.2792,-0.3762
Total Phosphorus,1203,33.0000,0.3191,0.3326,0.1029,0.3818,-0.2162
Nitrate,2663,15.0000,0.3984,4.4065,0.4119,4.3565,0.0136
Dissolved Oxygen,6458,20.0000,0.3249,2.1921,0.4744,1.9342,0.1495
Nitrate + Nitrite,971,33.0000,0.1896,3.9838,0.4579,3.2583,0.2683
Water Temperature,5892,27.0000,0.6403,5.3114,0.9417,2.1380,0.3014
pH,5451,28.0000,-0.0605,0.6540,0.3536,0.5106,0.4141
Nitrite,2225,15.0000,-0.6899,0.1605,-0.1211,0.1307,0.5688


### Test a single model on demand

Change `target` to re-run the evaluation for any one model.

In [11]:
target = "Water Temperature"
_ = test_model(target)

Water Temperature  (n_test=6,092 rows from 200 unseen stations)
    R2         = 0.9395
    RMSE       = 2.1685
    Error Rate = 23.05%


## Part 3 — Feature importances

`HistGradientBoostingRegressor` does not expose impurity-based importances, so we use **permutation importance**: each feature is shuffled on the test set and the resulting drop in R² measures how much the model relied on it. This is model-agnostic and reflects held-out performance. (It refits nothing but does re-score once per feature per repeat, so it takes a moment.)

In [12]:
def top_features(label: str, n: int = 10, n_repeats: int = 5) -> pd.DataFrame:
    """Return the n most important features by permutation importance on the test set."""
    entry = TRAINED[label]
    result = permutation_importance(
        entry["model"], entry["X_test"], entry["y_test"],
        scoring="r2", n_repeats=n_repeats, random_state=RANDOM_STATE, n_jobs=-1,
    )
    return (
        pd.DataFrame({
            "feature": FEATURE_COLS,
            "importance": result.importances_mean,
            "std": result.importances_std,
        })
        .sort_values("importance", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )


top_features("Water Temperature")

,feature,importance,std
0,doy_cos,0.1891,0.0027
1,prism_tmin_c,0.1215,0.0019
2,prism_tmax_c,0.0506,0.0008
3,doy,0.0278,0.0001
4,doy_sin,0.0239,0.0007
5,isu_max_feel_c,0.0215,0.0010
6,isu_min_feel_c,0.0137,0.0005
7,LatitudeMeasure,0.0105,0.0005
8,awc_mean,0.0104,0.0005
9,prism_tdmean_c,0.0098,0.0003


## Part 4 — Save trained models

Persist every fitted pipeline to `src/05_modeling/gradient_boosting/` as `gb_<target>.pkl`. Each file is self-contained (imputer + estimator) and can be reloaded with `pickle.load` for inference.

In [13]:
import pickle
import re

# Write .pkl files into src/05_modeling/gradient_boosting/ regardless of launch dir.
MODEL_DIR = REPO_ROOT / "src" / "05_modeling" / "gradient_boosting"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = "gb"  # gradient boosting

def target_stem(label: str) -> str:
    """'Nitrate + Nitrite' -> 'nitrate_nitrite', 'E. coli' -> 'e_coli'."""
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")

saved = []
for label, entry in TRAINED.items():
    path = MODEL_DIR / f"{PREFIX}_{target_stem(label)}.pkl"
    with open(path, "wb") as f:
        pickle.dump(entry["model"], f)
    saved.append(path.name)

print(f"Saved {len(saved)} models to {MODEL_DIR}:")
for name in saved:
    print("  ", name)

Saved 12 models to /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/src/05_modeling/gradient_boosting:
   gb_water_temperature.pkl
   gb_dissolved_oxygen.pkl
   gb_ph.pkl
   gb_nitrate.pkl
   gb_nitrite.pkl
   gb_nitrate_nitrite.pkl
   gb_total_phosphorus.pkl
   gb_specific_conductance.pkl
   gb_total_dissolved_solids.pkl
   gb_total_suspended_solids.pkl
   gb_turbidity.pkl
   gb_e_coli.pkl


## Part 5 — Publish metrics

Write this family's held-out results — including the station counts and the
persistence baseline — into `src/05_modeling/model_metrics.csv`, the table the
dashboard reads and `model_outcomes.md` summarises.

In [14]:
# --- Publish the metrics table read by app.py and model_outcomes.md ---
# Each notebook owns its own family's rows: we replace them in place and leave
# the other two families untouched, so the notebooks can be run in any order.
FAMILY = "Gradient Boosting"
FAMILY_ORDER = ["Linear Regression", "Random Forest", "Gradient Boosting"]
METRICS_PATH = REPO_ROOT / "src" / "05_modeling" / "model_metrics.csv"

rows = []
for label, entry in TRAINED.items():
    m = test_model(label, verbose=False)
    p = persistence_baseline(label)
    rows.append({
        "target": label,
        "model": FAMILY,
        "r2": round(m["R2"], 4),
        "rmse": round(m["RMSE"], 4),
        "mae": round(m["MAE"], 4),
        "error_rate": round(m["Error Rate (%)"], 2),
        "test_rows": entry["n_test"],
        "test_stations": entry["n_test_stations"],
        "train_rows": entry["n_train"],
        "train_stations": entry["n_train_stations"],
        "persistence_pairs": p["n_pairs"],
        "persistence_median_gap_days": p["median_gap_days"],
        "persistence_r2": round(p["Persistence R2"], 4),
        "model_r2_paired": round(p["Model R2 (paired)"], 4),
        "model_minus_persistence": round(p["Margin"], 4),
    })

family_metrics = pd.DataFrame(rows)

if METRICS_PATH.exists():
    existing = pd.read_csv(METRICS_PATH)
    if set(existing.columns) != set(family_metrics.columns):
        print("[WARN] existing model_metrics.csv uses the older schema — dropping its "
              "rows. Re-run the other two notebooks to refill them.")
        existing = existing.iloc[0:0]
    combined = pd.concat([existing[existing["model"] != FAMILY], family_metrics],
                         ignore_index=True)
else:
    combined = family_metrics

combined = (
    combined
    .assign(_f=pd.Categorical(combined["model"], FAMILY_ORDER, ordered=True),
            _t=pd.Categorical(combined["target"], list(TARGETS), ordered=True))
    .sort_values(["_f", "_t"])
    .drop(columns=["_f", "_t"])
)
combined.to_csv(METRICS_PATH, index=False)
print(f"Wrote {len(family_metrics)} {FAMILY} rows "
      f"({len(combined)} total) to {METRICS_PATH}")
family_metrics

Wrote 12 Gradient Boosting rows (36 total) to /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/src/05_modeling/model_metrics.csv


,target,model,r2,rmse,mae,error_rate,test_rows,test_stations,train_rows,train_stations,persistence_pairs,persistence_median_gap_days,persistence_r2,model_r2_paired,model_minus_persistence
0,Water Temperature,Gradient Boosting,0.9395,2.1685,1.6219,23.0500,6092,200,28613,799,5892,27.0000,0.6403,0.9417,0.3014
1,Dissolved Oxygen,Gradient Boosting,0.4668,1.9616,1.4071,16.6700,6640,182,25185,727,6458,20.0000,0.3249,0.4744,0.1495
2,pH,Gradient Boosting,0.3517,0.5101,0.3683,4.6900,5673,222,26680,885,5451,28.0000,-0.0605,0.3536,0.4141
3,Nitrate,Gradient Boosting,0.4155,4.3784,2.6519,105.2900,2719,56,9628,222,2663,15.0000,0.3984,0.4119,0.0136
4,Nitrite,Gradient Boosting,-0.1216,0.1297,0.0461,180.3700,2271,46,9340,183,2225,15.0000,-0.6899,-0.1211,0.5688
5,Nitrate + Nitrite,Gradient Boosting,0.4998,3.1969,2.1952,66.7600,1045,74,3614,293,971,33.0000,0.1896,0.4579,0.2683
6,Total Phosphorus,Gradient Boosting,0.0266,0.4705,0.2024,66.9900,1294,91,4592,362,1203,33.0000,0.3191,0.1029,-0.2162
7,Specific Conductance,Gradient Boosting,0.1846,183.3914,138.1098,28.1900,4617,97,11450,386,4520,1.0000,0.8559,0.1581,-0.6978
8,Total Dissolved Solids,Gradient Boosting,0.4382,94.2563,69.2902,22.5300,3902,117,14109,465,3785,28.0000,0.8107,0.4345,-0.3762
9,Total Suspended Solids,Gradient Boosting,-0.0357,212.2647,72.7747,101.6100,2435,111,12088,442,2324,31.0000,-0.9931,-0.0401,0.9530


---

**Notes**

* Gradient boosting and random forests are both tree ensembles. Under the old
  random row split boosting led on 9 of 12 targets; under the station-grouped
  split that reverses — random forest wins 9 of 12 and boosting's Specific
  Conductance collapses to 0.18 against the forest's 0.66. `early_stopping=True`
  validates on a random slice of the *training* stations, so it selects a
  stopping point against station-level structure that does not carry over to
  unseen sites. Try `early_stopping=False` with a fixed `max_iter`, or an
  explicit group-aware validation slice, before reading much into a GB score.
* The heavily right-skewed targets (nitrite, turbidity, *E. coli*) remain hard
  on the raw scale; a `log10` target transform is the natural next step if you
  want to push their error rates down (`src/04_eda/eda-summary.md` §4.3).
* To persist a fitted model, `pickle.dump(TRAINED[label]["model"], ...)`; the
  pipeline is self-contained (imputer + estimator).
* `GB_PARAMS` at the top controls the booster. `early_stopping=True` means
  `max_iter` is only an upper bound — the printed "rounds" is where each target
  actually stopped.
